# Lab 4 — Active Learning II: Diversity and Representative Sampling with Embeddings

In the previous lab, we used **uncertainty sampling** to select messages for annotation and evaluated whether it helped our classifier learn with fewer labels.

However, uncertainty is only one way to prioritize examples. The most uncertain messages may be very similar to each other, while other parts of the data remain poorly represented in the training set.

**In this lab, we will explore alternative acquisition strategies:** Can we select examples that expand our training data's coverage, avoid redundant queries, or represent different regions of the unlabeled pool?

### Extending Our Active-Learning Experiment

We will continue using the **BANKING77 intent-classification task** and the active-learning framework developed in Lab 3.

We will keep the same basic setup: a small labeled set, an unlabeled pool, a limited annotation budget, and a simulated annotation process. Instead of rebuilding the pipeline, we will focus on changing **how examples are selected for annotation**.

In this notebook, we will:

1. **Explore novelty sampling:** Select examples that are different from those already labeled.
2. **Explore diversity sampling:** Construct batches that avoid selecting very similar examples.
3. **Explore cluster-based sampling:** Use k-means to select representative examples from different regions of the pool.
4. **Explore relative representative sampling:** Use difference between labelled ($L_t$) and unlabelled ($U_t$) pool support.
5. **Compare acquisition strategies:** Examine which messages each strategy selects under the same annotation budget.
6. **Evaluate their impact:** Compare learning curves to investigate whether different selection strategies improve label efficiency.

### Embedding setup

Install Gensim and, if needed, spaCy's English model in the course environment:

```bash
pip install gensim spacy
python -m spacy download en_core_web_sm
```

From the repository root, download the pretrained GloVe model once:

```bash
python scripts/download_gensim.py glove-wiki-gigaword-50 models/
```

The model is stored in `models/glove-wiki-gigaword-50/`. The notebook sets `GENSIM_DATA_DIR` to the absolute path of `../../models` **before importing Gensim**. Run this notebook from `labs/Lab04/`.


## 1. Data Preparation

We continue using the **BANKING77** dataset, available on Hugging Face as `PolyAI/banking77`.

### 1.1 Loading the dataset

You should already have downloaded the dataset already in the previous lab. If not, download the dataset beforehand using the `download_dataset.py` script:

````bash
python scripts/download_dataset.py "PolyAI/banking77" "data/PolyAI_banking77"
````

In [ ]:
from datasets import load_dataset, load_from_disk

#ds = load_dataset("PolyAI/banking77"
ds = load_from_disk("../../data/PolyAI_banking77")

ds

### 1.2 Preparing the dataset

We continue with the same setting as in our baseline: we focus on **nine related intents** (out of 77) involving account access, transfers, card payments, and cash withdrawals.

We will also create a validation split, as the original dataset provides predefined training and test sets, but no validation set.

In [ ]:
import pandas as pd

# Labels are stored as integers; retrieve their names
label_names = ds["train"].features["label"].names

# Convert the training set to Pandas 
df_train = ds["train"].to_pandas()
df_train["intent"] = df_train["label"].map(lambda i: label_names[i])

# Convert the test set to Pandas as well
df_test = ds["test"].to_pandas()
df_test["intent"] = df_test["label"].map(lambda i: label_names[i])

# let's show a sample
display(df_train.sample(5, random_state=42))

In [ ]:
from sklearn.model_selection import train_test_split

# Original BANKING77 label -> course label
source_to_intent = {
    "passcode_forgotten": "account_access_passcode",
    "pending_transfer": "bank_transfer_pending",
    "transfer_not_received_by_recipient": "bank_transfer_not_received",
    "pending_card_payment": "card_payment_pending",
    "declined_card_payment": "card_payment_declined",
    "transaction_charged_twice": "card_payment_duplicated",
    "pending_cash_withdrawal": "cash_withdrawal_pending",
    "cash_withdrawal_not_recognised": "cash_withdrawal_unrecognized",
    "card_acceptance" : "card_acceptance"
}

# Original BANKING77 intents we want to keep
target_intents = list(source_to_intent.keys())

# Filter the datasets
df_train = df_train[df_train["intent"].isin(target_intents)].copy()
df_test = df_test[df_test["intent"].isin(target_intents)].copy()

# Rename the intents
df_train["intent"] = df_train["intent"].map(source_to_intent)
df_test["intent"] = df_test["intent"].map(source_to_intent)


SEED = 42

df_train_candidates, df_val = train_test_split(
    df_train,
    test_size=0.20,
    stratify=df_train["intent"],
    random_state=SEED
)

# Keep the official test set separate
print("Training candidates:", len(df_train_candidates))
print("Validation:", len(df_val))
print("Test (reserved):", len(df_test))

# Display class distribution
display(df_train["intent"].value_counts())


### 1.3 Creating the Active-Learning Pool

In active learning, we distinguish between two sets:

- **Labeled set ($L_0$)**: examples whose labels are available for training.
- **Unlabeled pool ($U_0$)**: examples whose labels are not yet available to the learner.

We will start with only **10 labeled examples per intent** (90 in total). All remaining training candidates will form our unlabeled pool.

Since BANKING77 is already labeled, we will **simulate annotation** by hiding the labels in the pool and revealing them only when the acquisition strategy selects an example.


### Preparing the Active-Learning Pool

As in Lab 3, we begin with a small labeled dataset ($L_0$) and treat the remaining training examples as an unlabeled pool ($U_0$).

We will compare several acquisition strategies using **the same initial labeled examples, unlabeled pool, classifier, and annotation budget**. This allows us to investigate how changing the acquisition strategy affects the selected examples and subsequent learning.

To make our experiments reproducible, we will use a function to create the initial labeled set and pool. We can also reuse it later to repeat experiments with different random seeds.

In [ ]:
def create_initial_pool(df_train_candidates, n_per_class=10, seed=42):
    """Create an initial labeled set and an unlabeled pool."""

    df_labeled = (
        df_train_candidates
        .groupby("intent", group_keys=False)
        .sample(n=n_per_class, random_state=seed)
        .copy()
    )

    df_pool = df_train_candidates.drop(df_labeled.index).copy()

    return df_labeled, df_pool


# Initial labeled examples per intent
N_INITIAL_PER_CLASS = 10

df_labeled, df_pool = create_initial_pool(
    df_train_candidates,
    n_per_class=N_INITIAL_PER_CLASS,
    seed=SEED
)

print("Initial labeled set:", len(df_labeled))
print("Unlabeled pool:", len(df_pool))
print("Validation set:", len(df_val))
print("Reserved test set:", len(df_test))

display(df_labeled["intent"].value_counts())



### 1.4 Defining the Annotation Budget

We cannot label the entire pool. Instead, we have a fixed budget of additional annotations.

For this experiment:

- **Initial labeled set:** 90 examples
- **Query batch size:** 20 examples per round
- **Annotation budget:** 200 additional labels
- **Active-learning rounds:** 10

We will keep this annotation budget fixed when comparing acquisition strategies.

Each strategy will select **20 messages per round**, for a total of **200 additional labels**. The difference will be *which messages are selected*, not how many labels are available.

As before, we will simulate human annotation by revealing the existing BANKING77 labels only after examples have been selected.

In [ ]:

# Active-learning configuration
BATCH_SIZE = 20
ANNOTATION_BUDGET = 200

N_ROUNDS = ANNOTATION_BUDGET // BATCH_SIZE

print("Initial labeled examples:", len(df_labeled))
print("Additional annotation budget:", ANNOTATION_BUDGET)
print("Query batch size:", BATCH_SIZE)
print("Number of rounds:", N_ROUNDS)
print("Final labeled set size:", len(df_labeled) + ANNOTATION_BUDGET)

assert ANNOTATION_BUDGET <= len(df_pool)
assert ANNOTATION_BUDGET % BATCH_SIZE == 0


## 2. From Uncertainty to Similarity-Based Selection

Last week, we selected messages using the classifier's **predicted probabilities**. We prioritized examples where the model was least certain.

This week, we will use the **relationships between messages** to guide selection.

For example, we might want to:

- Find messages that are different from those we have already labeled (**novelty**).
- Avoid selecting several nearly identical messages in the same batch (**diversity**).
- Select messages that represent different regions of the unlabeled pool (**cluster-based sampling**).
- Select examples that are prominent in the unlabelled pool and poorly covered in the labelled pool (**representative sampling**).

To do this, we need a way to measure similarity between messages.

### 2.1 Representing Messages with Word Embeddings

We will reuse the pretrained **GloVe word embeddings** from the embeddings seminar. For each message, we tokenize and lemmatize with spaCy, remove stopwords and punctuation, and average its known word vectors into one 50-dimensional document vector. We L2-normalize these vectors and compute them once for all training candidates, without using their intent labels.

We then use **cosine similarity** to compare messages. A value close to **1** indicates similar vector directions; a value close to **0** indicates nearly orthogonal vectors. Embeddings can relate words with different surface forms, but they do not necessarily capture every distinction relevant to banking intents.

The classifier later in the lab remains TF-IDF + logistic regression; only the similarity-based acquisition representation changes. These similarities provide the foundation for our new acquisition strategies.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import numpy as np
import os
import spacy

# Point Gensim to the downloaded model before importing it.
GENSIM_DATA_DIR = os.path.abspath("../../models")
os.environ["GENSIM_DATA_DIR"] = GENSIM_DATA_DIR
import gensim.downloader as api

model_name = "glove-wiki-gigaword-50"
word_vectors = api.load(model_name)
nlp = spacy.load("en_core_web_sm")

def tokenize(text):
    doc = nlp(text)
    return [t.lemma_.lower() for t in doc
            if not t.is_stop and not t.is_punct and t.is_alpha]

def doc_vector(text):
    # Mean-pool the known word vectors, as in VL06-Embeddings.
    vectors = [word_vectors[token] for token in tokenize(text)
               if token in word_vectors.key_to_index]
    if len(vectors) == 0:
        return np.zeros(word_vectors.vector_size)
    return np.mean(vectors, axis=0)

# Compute once; the DataFrame index keeps rows aligned as the pool changes.
candidate_vectors = np.vstack([
    doc_vector(text) for text in df_train_candidates["text"]
])
candidate_embeddings = pd.DataFrame(
    normalize(candidate_vectors, norm="l2"),
    index=df_train_candidates.index
)

# Retrieve the initial labeled set and pool in their current row order.
X_labeled = candidate_embeddings.loc[df_labeled.index].to_numpy()
X_pool = candidate_embeddings.loc[df_pool.index].to_numpy()

print("Labeled representations:", X_labeled.shape)
print("Pool representations:", X_pool.shape)
print("Zero vectors (no known words):", np.sum(np.linalg.norm(candidate_vectors, axis=1) == 0))

### 2.2 Inspecting Similar Messages
Before implementing an acquisition strategy, let's check what similarity looks like for actual BANKING77 messages.

We will select one labeled message and find its five most similar neighbors in the unlabeled pool.

In [ ]:
# Select a message from the labeled set
reference = df_labeled.iloc[0]

# Compare it with every message in the pool
similarities = cosine_similarity(
    X_labeled[0:1],
    X_pool
).ravel()

# Find the five most similar pool examples
nearest = np.argsort(similarities)[-5:][::-1]

print("Reference message:")
print(reference["text"])
print("\nMost similar unlabeled messages:")

results = df_pool.iloc[nearest][["text"]].copy()
results["similarity"] = similarities[nearest]

display(results)

**Questions**

1. Do the retrieved messages express similar requests?
2. Do they share similar words or phrases?
3. Could two messages express the same intent but receive relatively low embedding similarity?

Remember that **similarity depends on the representation**. Mean word embeddings capture some relationships between different words, but may miss task-relevant distinctions.

We can now use these representations to decide which examples to query. Our first strategy will prioritize messages that are **least similar to the examples we have already labeled**.

## 3. Novelty Sampling 
Uncertainty sampling prioritizes examples the model finds difficult to classify. But an example can be unfamiliar even when the model predicts its label confidently.

**Novelty sampling** takes a different approach: it prioritizes examples that are different from those we have already labeled.

For each message in the unlabeled pool, we:

1. Find its most similar message in the labeled set.
2. Measure how different the two messages are.
3. Prefer candidates with the highest novelty scores.

We will use cosine distance, defined as:

$$
d(x,z)=1-\operatorname{cosine\_similarity}(x,z)
$$

The novelty score of a candidate is its distance to the **nearest labeled example**:

$$
a_{\text{novelty}}(x)=\min_{l\in L_t}d(x,l)
$$

A larger score means that even the nearest labeled example is relatively different from the candidate.

### 3.1 Implementing Novelty Sampling
We already have the normalized document embeddings of our labeled set and unlabeled pool.

To find the nearest labeled neighbor using cosine distance, we can find the highest cosine similarity and subtract it from 1.

In [ ]:
def acquire_novelty(X_labeled, X_pool, batch_size):
    """Select the most novel pool examples relative to the labeled set."""

    # Compare every pool example with the labeled examples
    similarities = cosine_similarity(X_pool, X_labeled)

    # Similarity to the nearest labeled neighbor
    nearest_similarity = similarities.max(axis=1)

    # Higher distance means greater novelty
    novelty_scores = 1 - nearest_similarity

    # Select the most novel examples
    selected = np.argsort(-novelty_scores)[:batch_size]

    return selected


# Select a batch using novelty sampling
novelty_selected = acquire_novelty(
    X_labeled,
    X_pool,
    BATCH_SIZE
)

print("Selected examples:", len(novelty_selected))

### 3.2 Inspecting the Selected Messages
Now let's inspect which messages novelty sampling prioritizes and how similar they are to the labeled examples.

In [ ]:
# Compute novelty scores for inspection
similarities = cosine_similarity(X_pool, X_labeled)
nearest_labeled = similarities.argmax(axis=1)
novelty_scores = 1 - similarities.max(axis=1)

# Display the selected examples
novelty_results = df_pool.iloc[novelty_selected][["text"]].copy()

novelty_results["novelty"] = novelty_scores[novelty_selected]

novelty_results["nearest_labeled_message"] = (
    df_labeled.iloc[
        nearest_labeled[novelty_selected]
    ]["text"].to_numpy()
)

display(novelty_results.round(3))

#### Interpreting the Selection

Examine the messages with the highest novelty scores.

**Questions**

1. Do the selected messages express requests that are different from those already in the labeled set?
2. Are some messages unusual because they contain rare words, unexpected expressions, or very little text?
3. Would all these messages necessarily be useful to label?

**Key observation:** Novelty identifies examples that are different from what we have already labeled. It does not tell us *why* they are different or whether labeling them will improve the classifier.

### 3.3 What About the Whole Batch?

Our novelty function ranks each candidate independently, according to its distance from the labeled set.

But consider two unlabeled messages that are both very different from everything we have labeled, yet very similar to each other.

Both may receive high novelty scores and be selected in the same batch.

**Novelty does not automatically prevent redundant queries.**

Next, we will change our selection rule: instead of asking only how different a candidate is from the labeled set, we will also consider how different it is from the examples already selected for annotation.

## 4. Diversity Sampling 

In novelty sampling, each message receives a score based on its distance from the **labeled set**.

But imagine that several highly novel messages are also very similar to each other. Selecting all of them might spend much of our annotation budget on essentially the same kind of request.

**Diversity sampling** considers the examples selected for a batch together. Instead of selecting the highest-scoring messages independently, we build a batch in which each new example adds coverage.

### 4.1 Greedy Farthest-First Selection

One way to construct a diverse batch is **farthest-first selection**:

1. Start with one selected example.
2. For each remaining candidate, measure its distance to the **nearest example already selected**.
3. Select the candidate with the largest such distance.
4. Repeat until the batch contains $k$ examples.

For a candidate $x$ and the current query batch $Q$:

$$
d(x,Q)=\min_{q\in Q}d(x,q)
$$

At each step, we choose the candidate that maximizes this distance.

**The key difference:** Novelty measures distance from the labeled set ($L_t$). Farthest-first diversity measures distance from the growing query batch ($Q$).

In [ ]:
def acquire_diversity(X_labeled, X_pool, batch_size):
    """Build a diverse batch using greedy farthest-first selection."""

    if not 1 <= batch_size <= X_pool.shape[0]:
        raise ValueError("Invalid batch size.")

    # Start with the most novel example
    first = int(acquire_novelty(X_labeled, X_pool, 1)[0])
    selected = [first]

    # Distance from each candidate to the selected batch
    min_distances = (
        1 - cosine_similarity(X_pool, X_pool[first:first + 1]).ravel()
    )
    # Exclude example already selected
    min_distances[first] = -np.inf

    # Add the most distant candidate at each step
    while len(selected) < batch_size:

        next_idx = int(np.argmax(min_distances))
        selected.append(next_idx)

        # Distance to the newly selected example
        new_distances = (
            1 - cosine_similarity(
                X_pool, X_pool[next_idx:next_idx + 1]
            ).ravel()
        )

        # For each pool example, keep its distance to the nearest selected example.
        # Compare its current minimum distance (to the examples already in Q)
        # with its distance to the newly selected example.
        # Keep the smaller value, since the new example may now be its closest
        # selected neighbor.
        min_distances = np.minimum(
            min_distances, new_distances
        )

        # Exclude examples already selected
        min_distances[selected] = -np.inf

    return np.array(selected)


diversity_selected = acquire_diversity(
    X_labeled,
    X_pool,
    BATCH_SIZE
)

print("Selected examples:", len(diversity_selected))

The central operation is:

````python
min_distances = np.minimum(min_distances, new_distances)
````

Each time we add an example to the batch, we update every remaining candidate's distance to its nearest selected neighbor.

This is what makes the selection dependent on the batch we are constructing.

### 4.2 Inspecting the Diverse Batch
We can now examine the selected messages, in the order the algorithm chose them.

Compare the messages selected by **novelty sampling** in the previous section with those selected by **farthest-first diversity**.

In [ ]:
diversity_results = df_pool.iloc[
    diversity_selected
][["text"]].copy()

diversity_results.insert(
    0, "selection_order",
    np.arange(1, len(diversity_results) + 1)
)

display(diversity_results)

**Questions**

1. Are the selected messages noticeably different between the two strategies?
2. Do you see repeated or very similar messages within either batch?
3. Why might farthest-first select a message that has a lower novelty score than another available candidate?

Notice that farthest-first does not necessarily select the messages that are most different from our existing labeled data. Its goal is to spread the *newly selected examples* across the representation space.

### 4.3 Measuring Batch Redundancy
We have inspected the selected messages manually. Now let's compare their similarity more systematically.

A simple measure of redundancy is the average pairwise cosine similarity between the messages in a batch.

If the selected messages are very similar to one another, their average similarity will generally be higher.

![Measuring batch redundancy](img/similarity_batch.svg)

In [ ]:
def average_batch_similarity(X_pool, selected):
    """Average cosine similarity between distinct batch members."""

    X_batch = X_pool[selected]

    similarities = cosine_similarity(X_batch)

    # Use each pair only once, excluding the diagonal
    upper = np.triu_indices(len(selected), k=1)

    return similarities[upper].mean()


novelty_similarity = average_batch_similarity(
    X_pool, novelty_selected
)

diversity_similarity = average_batch_similarity(
    X_pool, diversity_selected
)

print(f"Novelty batch:   {novelty_similarity:.3f}")
print(f"Diversity batch: {diversity_similarity:.3f}")

#### Interpreting Batch Similarity

**Questions**

1. Which batch has the lower average pairwise similarity?
2. Does that match your impression from reading the selected messages?
3. Does lower similarity necessarily mean the selected examples will improve the classifier more?

**Important:** Farthest-first sampling explicitly avoids selecting examples close to those already in the batch. However, greater geometric diversity does not necessarily mean better learning.

For example, the algorithm may prefer an isolated message simply because it is different from everything else, even if that message represents a rare or unhelpful pattern.

## 5. Cluster-Based Representative Sampling

Novelty and farthest-first sampling use distances to prioritize examples. But selecting distant examples does not necessarily mean selecting messages that represent common patterns in our data.

An alternative is to first group similar messages into **clusters** and then select a representative from each cluster.

We will use **k-means** to construct our next annotation batch:

1. Divide the unlabeled pool into clusters using its document embeddings.
2. Find each cluster's centroid.
3. Select the actual message closest to each centroid.
4. Combine these messages into an annotation batch.

A centroid represents the center of a cluster, but it is not necessarily an actual message. Therefore, we must select a real example from the pool.

For simplicity, we will create **one cluster per available label**:

$$
m=k=20
$$

Here, $m$ is the number of clusters and $k$ is our annotation batch size.

### 5.1 Implementing K-Means-Based Sampling
We will use the same document embeddings as before. K-means groups the pool into clusters, and Euclidean distance determines which example is closest to each centroid.

Because our nonzero document embeddings are L2-normalized, Euclidean distance and cosine similarity give consistent orderings when comparing individual document vectors. K-means itself still minimizes Euclidean distances to centroids.

**Note on embedding normalization**

We apply **L2 normalization** to each nonzero document embedding: each vector is scaled to have length 1, while preserving its direction. A message with no known words remains a zero vector.

For L2-normalized vectors, Euclidean distance and cosine similarity are directly related:

$$
\|\mathbf{x}-\mathbf{y}\|^2 = 2(1-\cos(\mathbf{x},\mathbf{y}))
$$

This means that **comparing document vectors using Euclidean distance or cosine similarity produces the same ranking** (closest to farthest).

**Sampling choice**: K-means identifies the clusters, but we still need to decide which examples to query. We could select a random example, the closest to the centroid (representative), or the farthest (peripheral). Here, we choose the closest to prioritize representative examples.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances

def acquire_kmeans(X_pool, batch_size, seed=42, return_info=False):
    """Select one centroid-nearest example from each cluster."""

    # Create one cluster per available label
    kmeans = KMeans(
        n_clusters=batch_size,
        random_state=seed,
        n_init=5
    )
    cluster_labels = kmeans.fit_predict(X_pool)

    # Distance from every pool example to each centroid
    distances = euclidean_distances(
        X_pool, kmeans.cluster_centers_
    )

    selected = []

    # Sampling strategy: select the example closest to each cluster's centroid.
    # Other options include random sampling or selecting the farthest example.
    # Here, we prioritize representative (central) examples.
    for cluster_id in range(batch_size):
        members = np.where(cluster_labels == cluster_id)[0]

        nearest = members[
            np.argmin(distances[members, cluster_id])
        ]

        selected.append(nearest)

    selected = np.array(selected)

    if return_info:
        cluster_sizes = np.bincount(
            cluster_labels, minlength=batch_size
        )
        return selected, cluster_labels, cluster_sizes, distances

    return selected

Notice that the algorithm makes two decisions: k-means determines which examples belong together, and the centroid-nearest rule determines which actual message represents each group.

Now let's apply the function.

In [ ]:
kmeans_selected, cluster_labels, cluster_sizes, distances = (
    acquire_kmeans(
        X_pool,
        BATCH_SIZE,
        seed=SEED,
        return_info=True
    )
)

print("Number of clusters:", len(cluster_sizes))
print("Selected examples:", len(kmeans_selected))
print("Cluster sizes:", cluster_sizes)

### 5.2 Inspecting a Cluster

K-means has divided our unlabeled messages into groups based on their document embeddings.

But what do these clusters actually contain?

Let's inspect the **largest cluster** and read a few messages assigned to it.

In [ ]:
# Inspect the largest cluster
cluster_id = int(np.argmax(cluster_sizes))
members = np.where(cluster_labels == cluster_id)[0]

print("Cluster:", cluster_id)
print("Size:", len(members))
print("Selected representative:")
print(df_pool.iloc[kmeans_selected[cluster_id]]["text"])

print("\nOther messages in this cluster:")
display(
    df_pool.iloc[members][["text"]]
    .sample(n=min(8, len(members)), random_state=SEED)
)

**Questions**

- What do these messages have in common?
- Do they seem to express the same banking intent, or do they mainly share words and expressions?
- Which message would you choose to represent this cluster?

### 5.3 Inspecting Cluster Representatives
We can now inspect the selected messages, their cluster sizes, and their distances to the corresponding centroids.

In [ ]:

cluster_results = pd.DataFrame({
    "cluster": np.arange(BATCH_SIZE),
    "cluster_size": cluster_sizes,
    "representative_message": (
        df_pool.iloc[kmeans_selected]["text"].to_numpy()
    ),
    "distance_to_centroid": distances[
        kmeans_selected,
        np.arange(BATCH_SIZE)
    ]
})

display(cluster_results.round(3))


**Questions**

1. Are some clusters much larger than others?
2. Do the selected messages look typical of the patterns they might represent?
3. Do you expect these clusters to correspond directly to the banking intents?

Remember that **clusters are not necessarily classes**. K-means groups messages based on their document embeddings, not their intent labels.

Also, selecting one representative per cluster gives every cluster one label, regardless of how many messages it contains.

### 5.4 Comparing Our Three Geometric Strategies
We have now implemented three different strategies that use the relationships between examples:
- **Novelty**: Select examples distant from the labeled set.
- **Farthest-first diversity**: Spread the selected examples apart.
- **K-means representatives**: Select typical examples from different clusters.
  
Let's compare the redundancy of their selected batches using the function we developed in the previous section.

In [ ]:
batch_comparison = pd.DataFrame({
    "Strategy": [
        "Novelty",
        "Farthest-first",
        "K-means representatives"
    ],
    "Average pairwise similarity": [
        average_batch_similarity(X_pool, novelty_selected),
        average_batch_similarity(X_pool, diversity_selected),
        average_batch_similarity(X_pool, kmeans_selected)
    ]
})

display(batch_comparison.round(3))

#### Interpreting the Comparison

**Questions**

1. Which strategy produced the batch with the lowest average pairwise similarity?
2. Did k-means produce a less redundant batch than novelty sampling?
3. Why might k-means select examples that are closer together than those selected by farthest-first?

These methods optimize different properties. K-means representatives aim to select typical examples from different clusters, while farthest-first explicitly prioritizes separation between selected examples.

**Important:** Different strategies optimize different properties of the data. Farthest-first explicitly spreads the selected examples apart, novelty prioritizes examples unfamiliar relative to the labeled set, and k-means selects representatives of different clusters.

A less redundant batch is not necessarily a more useful batch. To investigate usefulness, we need to evaluate what happens after the selected examples are labeled and the classifier is retrained.


## 6. Representative Sampling — Common in the Pool, Missing from Training

In the previous section, we used clustering to select examples from different regions of the unlabeled pool.

But selecting examples from different regions does not tell us whether those regions are **already well represented in our labeled data**.

Consider two situations:

- A pattern is common in the unlabeled pool, but we already have many similar labeled examples.
- A pattern is common in the unlabeled pool, but we have very few similar labeled examples.

The second situation may be more interesting for annotation.

**Representative sampling** can help us identify these differences by comparing how well each candidate is supported by two datasets:

1. **Pool support:** How similar is the candidate to other unlabeled messages?
2. **Labeled support:** How similar is the candidate to messages we have already labeled?

We will use average cosine similarity to estimate both:

$$
\rho_U(x) = \frac{1}{|U_t|-1}
\sum_{\substack{u \in U_t \\ u \neq x}}
\operatorname{sim}(x,u)
$$

$$
\rho_L(x) = \frac{1}{|L_t|}
\sum_{l \in L_t}\operatorname{sim}(x,l)
$$

We then calculate a **relative representativeness score**:

$$
a_{\mathrm{rep}}(x) = \rho_U(x) - \rho_L(x)
$$

A high score means that an example has greater similarity-based support in the unlabeled pool than in the labeled set.

### 6.1 Implement Relative Representativeness

We will reuse our normalized document embeddings to calculate the support scores efficiently.

Instead of comparing every message individually with all other messages, we compute the **mean document embedding** of the labeled set and the unlabeled pool.

We then:

1. Use the dot product with each mean vector to calculate average cosine similarity.
2. Exclude self-similarity when calculating pool support.
3. Subtract labeled support from pool support.
4. Select the `BATCH_SIZE` messages with the highest scores.

This avoids computing large pairwise similarity matrices. The function also returns the intermediate scores so we can inspect why particular messages were selected.


In [ ]:

def acquire_representative(
    X_labeled, X_pool, batch_size, return_scores=False
):
    """
    Select pool examples that are better represented in the
    unlabeled pool than in the existing labeled set.

    For each candidate, compute:
        score = pool_support - labeled_support

    Higher scores indicate a larger representation gap.

    Assumes L2-normalized nonzero document embeddings, so the dot product
    between two document vectors is their cosine similarity.
    """

    # Number of examples currently in the unlabeled pool
    n = X_pool.shape[0]

    # Check that we have enough examples for the calculations
    assert n > 1
    assert X_labeled.shape[0] > 0
    assert 1 <= batch_size <= n

    # STEP 1: Compute the mean vector of each dataset
    #
    # Each row is a document embedding of one message.
    # Taking the mean gives us an average feature vector
    # for the unlabeled pool and for the labeled set.
    #
    # .ravel() converts the result into a 1D array.

    pool_mean = np.asarray(X_pool.mean(axis=0)).ravel()
    labeled_mean = np.asarray(X_labeled.mean(axis=0)).ravel()

    # STEP 2: Calculate support in the unlabeled pool
    #
    # X_pool @ pool_mean computes the dot product between
    # each pool message and the mean pool vector.
    #
    # Because our nonzero document embeddings are L2-normalized, this
    # equals the average cosine similarity between each
    # candidate and ALL pool examples (including itself).

    pool_support = np.asarray(
        X_pool @ pool_mean
    ).ravel()

    # Exclude self-similarity from the average.
    #
    # Every nonzero embedding has cosine similarity 1.0 with itself.
    # We want similarity to the OTHER pool messages only.
    #
    # Multiplying by n recovers the sum of similarities.
    # Subtracting 1 removes the candidate's self-similarity.
    # Dividing by n-1 gives the average over the remaining
    # pool examples.

    # Zero vectors remain zero after normalization; for those rare messages
    # this approximation subtracts 1 even though their self-similarity is 0.
    pool_support = (n * pool_support - 1) / (n - 1)

    # STEP 3: Calculate support in the labeled set
    #
    # For each pool candidate, compute its average cosine
    # similarity to the messages we have already labeled.
    #
    # There is no self-similarity to remove here because
    # the candidate is not part of the labeled set.

    labeled_support = np.asarray(
        X_pool @ labeled_mean
    ).ravel()

    # STEP 4: Compare the two support scores
    #
    # Positive score:
    #   The candidate is more similar, on average,
    #   to the unlabeled pool than to the labeled set.
    #
    # Negative score:
    #   The candidate is more similar, on average,
    #   to the existing labeled examples.
    #
    # A high score suggests a possible representation gap,
    # but does not guarantee that the example is informative.

    scores = pool_support - labeled_support

    # STEP 5: Select the highest-scoring examples
    #
    # np.argsort(-scores) sorts candidate positions from
    # highest to lowest score.
    #
    # [:batch_size] keeps only the first batch_size examples.
    # We return positions in X_pool, not the messages themselves.

    selected = np.argsort(-scores)[:batch_size]

    # Optionally return the intermediate scores so we can
    # inspect and explain why particular messages were selected.

    if return_scores:
        return selected, pool_support, labeled_support, scores

    return selected


# Apply the acquisition function to our current datasets.
# Request the intermediate scores for inspection.

(
    representative_selected,
    pool_support,
    labeled_support,
    representative_scores
) = acquire_representative(
    X_labeled,
    X_pool,
    BATCH_SIZE,
    return_scores=True
)

print("Selected examples:", len(representative_selected))


### 6.2 Inspecting Representative Examples

Let's inspect the messages selected by relative representativeness.

For each example, we will examine:

- Its average similarity to the unlabeled pool.
- Its average similarity to the labeled set.
- The difference between these two scores.

**Before inspecting the results:** What would you expect to see among the highest-ranked messages?

In [ ]:

representative_results = df_pool.iloc[
    representative_selected
][["text"]].copy()

representative_results["pool_support"] = (
    pool_support[representative_selected]
)

representative_results["labeled_support"] = (
    labeled_support[representative_selected]
)

representative_results["difference"] = (
    representative_scores[representative_selected]
)

display(representative_results.head(10).round(3))


### 6.3 Interpreting Relative Representativeness

Examine the selected messages and their support scores.

**Questions**

1. Do the selected messages have greater support in the unlabeled pool than in the labeled set?
2. Do they appear to represent recurring banking requests that are missing or underrepresented in our labeled examples?
3. Could a message receive a high relative score even if its absolute pool support is fairly low?

**Key observation:** Relative representativeness does not simply look for unfamiliar messages. It prioritizes messages that are better supported by the unlabeled pool than by the existing labeled set.

However, this is only an approximation based on document-embedding similarity. A positive difference does not guarantee that the message represents an important missing pattern.


### 6.4 Comparing the Selected Batches

We now have four acquisition strategies that use similarity and distance, each prioritizing a different property:

- **Novelty:** Examples different from the existing labeled set.
- **Farthest-first:** Examples different from one another within the selected batch.
- **K-means representatives:** Typical examples from different clusters.
- **Relative representativeness:** Examples with greater similarity-based support in the unlabeled pool than in the labeled set.

Let's compare the **average pairwise similarity** of the batches selected by these strategies.

In [ ]:
batch_comparison = pd.DataFrame({
    "Strategy": [
        "Novelty",
        "Farthest-first",
        "K-means representatives",
        "Relative-representativeness"
    ],
    "Average pairwise similarity": [
        average_batch_similarity(X_pool, novelty_selected),
        average_batch_similarity(X_pool, diversity_selected),
        average_batch_similarity(X_pool, kmeans_selected),
        average_batch_similarity(X_pool, representative_selected)
    ]
})

display(batch_comparison.round(3))

**Questions**

1. Which strategy produces the lowest average pairwise similarity? Is this consistent with its selection objective?
2. Relative representativeness produces a much more similar batch. Looking at the selected messages, what explains this result?
3. Why does k-means select a more diverse batch than relative representativeness, even though both aim to improve representation?
4. Does a less redundant batch necessarily provide more useful training examples? How could we find out?

**Important** Selecting individually representative examples does not guarantee a diverse batch. Relative representativeness may select many similar examples from an underrepresented pattern, while k-means spreads selections across different clusters.

However, **neither diversity nor representativeness guarantees better classification performance**.

In the next section, we will evaluate whether these different selections actually help the classifier learn by simulating annotation, retraining the model, and comparing learning curves.


## 7. Evaluating Acquisition Strategies

We have seen how different acquisition strategies select different messages from the same unlabeled pool.

But do these differences actually help the classifier learn?

As in Week 3, we will simulate the active-learning process:

1. Train a classifier on the currently labeled messages.
2. Evaluate its performance on the validation set.
3. Select a batch of unlabeled messages using an acquisition strategy.
4. Reveal their labels and add them to the training set.
5. Repeat until the annotation budget is exhausted.

We will compare six strategies:

- **Random:** Select messages at random.
- **Least confidence:** Select messages the classifier is most uncertain about.
- **Novelty:** Select messages different from the labeled set.
- **Farthest-first:** Select a diverse batch.
- **K-means:** Select representatives from different clusters.
- **Relative representativeness:** Select messages with greater support in the pool than in the labeled set.

To make the comparison fair, all strategies use the same initial labeled set, classifier, validation set, batch size, and annotation budget.

**Question:** Which strategy will achieve the best classification performance with the same number of labels?


### 7.1 Connecting Our Acquisition Functions
Our existing functions use slightly different inputs. We will create one interface that selects the appropriate strategy.

In [ ]:
def select_batch(strategy, model, labeled, pool, batch_size, seed):
    """Select a batch and return its row positions in the current pool."""

    # BASELINES

    if strategy == "random":
        # Choose random pool positions without replacement
        rng = np.random.default_rng(seed)
        return rng.choice(len(pool), size=batch_size, replace=False)

    if strategy == "least_confidence":
        # Predict class probabilities for the current pool
        probabilities = model.predict_proba(pool["text"])

        # Lower maximum probability means greater uncertainty
        uncertainty = 1 - probabilities.max(axis=1)

        return np.argsort(-uncertainty)[:batch_size]

    # GEOMETRIC STRATEGIES
    # Retrieve the precomputed embeddings by original DataFrame index.
    # This keeps vectors aligned when the labeled set and pool change.

    X_labeled_now = candidate_embeddings.loc[labeled.index].to_numpy()
    X_pool_now = candidate_embeddings.loc[pool.index].to_numpy()

    if strategy == "novelty":
        return acquire_novelty(
            X_labeled_now, X_pool_now, batch_size
        )

    if strategy == "farthest_first":
        return acquire_diversity(
            X_labeled_now, X_pool_now, batch_size
        )

    if strategy == "kmeans":
        return acquire_kmeans(
            X_pool_now, batch_size, seed=seed
        )

    if strategy == "representative":
        return acquire_representative(
            X_labeled_now, X_pool_now, batch_size
        )

    raise ValueError(f"Unknown strategy: {strategy}")


### 7.2 Running the Active-Learning Loop
We can reuse the Week 3 evaluation procedure. Each experiment starts from fresh copies of the same labeled set and pool.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def run_active_learning(strategy, initial_labeled, initial_pool, seed):
    """Run active learning with one acquisition strategy."""

    # All strategies start with the same data
    labeled = initial_labeled.copy()
    pool = initial_pool.copy()
    history = []

    for round_idx in range(N_ROUNDS + 1):

        # 1. Train a new classifier on the current labeled set
        model = make_pipeline(
            TfidfVectorizer(),
            LogisticRegression(max_iter=1000, random_state=SEED)
        )
        model.fit(labeled["text"], labeled["intent"])

        # 2. Evaluate on the unchanged validation set
        predictions = model.predict(df_val["text"])

        history.append({
            "strategy": strategy,
            "round": round_idx,
            "n_labeled": len(labeled),
            "accuracy": accuracy_score(
                df_val["intent"], predictions
            ),
            "macro_f1": f1_score(
                df_val["intent"], predictions, average="macro"
            )
        })

        # Stop when the annotation budget is exhausted
        if round_idx == N_ROUNDS:
            break

        # 3. Select the next batch from the current pool
        selected = select_batch(
            strategy, model, labeled, pool,
            BATCH_SIZE, seed=SEED + round_idx
        )

        # 4. Simulate human annotation
        # The hidden labels are revealed only after selection
        queried = pool.iloc[selected]

        # 5. Update the labeled set and remove queried examples
        labeled = pd.concat([labeled, queried])
        pool = pool.drop(queried.index)

    return pd.DataFrame(history)

### 7.3 Comparing Learning Curves

We will now run all six strategies with the same annotation budget.

Each curve shows how validation macro-F1 changes as more messages are labeled.

Look particularly at the early rounds: a strategy that reaches good performance with fewer labels may be more useful even if other strategies eventually catch up.

In [ ]:
import matplotlib.pyplot as plt

strategies = [
    "random",
    "least_confidence",
    "novelty",
    "farthest_first",
    "kmeans",
    "representative"
]

N_REPEATS = 10
SEED = 42

all_results = []

for repeat in range(N_REPEATS):
    seed = SEED + repeat
    
    initial_labeled, initial_pool = create_initial_pool(
        df_train_candidates,
        n_per_class=N_INITIAL_PER_CLASS,
        seed=seed
    )

    for strategy in strategies:
        print(f"Seed {seed}: {strategy}")

        run_results = run_active_learning(
            strategy, initial_labeled, initial_pool, seed
        )

        run_results["seed"] = seed
        all_results.append(run_results)

results = pd.concat(all_results, ignore_index=True)

print("All experiments completed.")

In [ ]:
# Calculate mean and standard deviation for each strategy
# at each labeled-set size
df_summary = (
    results
    .groupby(["strategy", "n_labeled"])["macro_f1"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values(["strategy", "n_labeled"])
)

display(df_summary.head(10))

In [ ]:
plt_multiple_runs = plt.figure(figsize=(9, 5))

# Plot the results for each acquisition strategy
for strategy in strategies:
    subset = df_summary[df_summary["strategy"] == strategy]

    # Get the number of labels, mean performance, and standard deviation
    x = subset["n_labeled"].to_numpy()
    mean = subset["mean"].to_numpy()
    std = subset["std"].fillna(0).to_numpy()

    # Plot the mean learning curve
    line, = plt.plot(
        x, mean,
        marker="o",
        label=strategy.replace("_", " ").title()
    )

    # Shade the region within one standard deviation of the mean
    # Use the same color as the corresponding learning curve
    plt.fill_between(
        x,
        mean - std,
        mean + std,
        color=line.get_color(),
        alpha=0.15
    )

# Format the plot
plt.xlabel("Number of labeled examples")
plt.ylabel("Validation macro-F1")
plt.title("Active Learning: Mean Performance Across Repetitions")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Find the largest labeled-set size (final annotation budget)
final_n_labeled = results["n_labeled"].max()

# Keep only the final evaluation from each run
final_runs = results[
    results["n_labeled"] == final_n_labeled
]

# Calculate mean and standard deviation across runs
final_results = (
    final_runs
    .groupby("strategy")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std")
    )
    .sort_values("macro_f1_mean", ascending=False)
    .reset_index()
)

display(final_results.round(3))



### 7.5 Interpreting the Learning Curves

We repeated the active-learning experiment using different initial labeled sets. Each strategy used the same annotation budget and evaluation procedure.

**Questions**

1. Which strategy achieves the highest mean validation macro-F1? How early does its advantage become visible?
2. Compare least confidence with random sampling. Approximately how many labels does each need to reach a similar level of performance?
3. Farthest-first produced the most diverse batch in our earlier comparison. Does this translate into better classification performance?
4. Relative representativeness selected several highly similar messages. How does its learning curve compare with random sampling?
5. What do the shaded regions tell us about the stability of the results across runs?

**Key takeaway:** Greater diversity or representativeness does not necessarily translate into greater label efficiency.

Compare least-confidence sampling with the geometric strategies in your results. The outcome depends on our dataset, acquisition representation, classifier, and initial labeled data.

**Reflection:** Under what circumstances might diversity or representative sampling become more valuable than uncertainty sampling?

## 8. Applying Diversity and Representative Sampling to Your Project

Now it's your turn! Extend the active-learning experiments in your semester project with **diversity-based and representative sampling**.

In the previous lab, you implemented uncertainty sampling to identify examples your model finds difficult. This week, we explored a different perspective: selecting examples based on their relationships to the labeled set, the unlabeled pool, and other selected examples.

Your goal is to investigate whether **diversity or representativeness can improve label efficiency** in your project.

You do not need to implement every acquisition strategy covered in this notebook. Select **at least one diversity-based and one representativeness-based approach** that make sense for your task.

### 8.1 Choose a Suitable Data Representation

Diversity and representative sampling depend on how we measure similarity between examples.

In this lab, we used mean-pooled word embeddings and cosine similarity for short banking messages. However, your project may involve different data or models.

**1. Identify a suitable representation.**

Consider how your examples can be represented numerically:

- **Text:** TF-IDF vectors, sentence embeddings, or representations from your existing model.
- **Images:** Image embeddings or feature vectors from a pretrained model.
- **Structured data:** Numerical and encoded categorical features, with appropriate scaling.
- **Other tasks:** Task-specific feature representations that capture meaningful similarities.

You may reuse representations already available in your project.

This also mean that you need to use a similary measure that is suitable for your data. 

Remember that **the quality of similarity-based acquisition depends on the quality of the representation**.

### 8.2 Implement Diversity-Based Sampling

Extend your active-learning notebook with **at least one diversity-based acquisition function**.

You may choose an approach demonstrated in this lab or adapt another suitable method.

Possible approaches include:

- **Novelty sampling:** Prioritize examples that are distant from the current labeled set.
- **Farthest-first sampling:** Select examples that are different from one another within the acquisition batch.
- **Cluster-based sampling:** Group examples in the unlabeled pool and select representatives from different clusters.

You do not need to implement all three.

**Explain your choice:**

1. What does diversity mean for your task?
2. How does your acquisition function measure it?

Implement the acquisition function so it can be used within your existing active-learning loop.

### 8.3 Implement Representative Sampling

Next, implement **at least one representativeness-based acquisition function**.

Unlike diversity sampling, which aims to explore different parts of the data space, representative sampling considers how well an example reflects patterns in the unlabeled pool.

You may choose, for example:

- **Relative representativeness:** Compare similarity-based support in the unlabeled pool with support in the existing labeled set, prioritizing patterns that appear underrepresented in the labeled data.

Choose an approach that is meaningful for your project. If you expect differences between your labeled data and the data your system will eventually encounter, relative representativeness may be particularly interesting.

**Explain your choice:**

1. How do you estimate whether an example is representative?
2. Does your approach consider the existing labeled set, the unlabeled pool, or both?

### 8.4 Integrate and Compare Acquisition Strategies

Extend the active-learning experiment you developed in the previous lab.

Your comparison should now include:

- **Random sampling:** Your baseline.
- **Uncertainty sampling:** The acquisition function implemented in Week 3.
- **Diversity-based sampling:** Your selected diversity strategy.
- **Representative sampling:** Your selected representativeness strategy.

Reuse your existing model, annotation simulation, and evaluation procedure.

To ensure a fair comparison, use the same initial labeled set, acquisition batch size, total annotation budget, and evaluation data across strategies.


### 8.5 Evaluate Label Efficiency

Reuse the evaluation framework from your previous active-learning experiment.

- Run multiple acquisition rounds, retraining or updating your model as new annotations become available.
- Evaluate performance after each round using metrics appropriate to your project.
- Compare learning curves for random, uncertainty, diversity, and representative sampling.
- Repeat the experiment with different random seeds or initial labeled sets.
- Report mean performance and variability across repetitions.

You do not need to create a new evaluation framework. **Extend the one you already implemented.**

### 8.6 Analyze Your Results

Discuss the following questions:

1. **Representation:** What feature representation and similarity measure did you choose? How well do they capture meaningful relationships in your data?
2. **Selection behavior:** What kinds of examples do your diversity and representative sampling strategies select?
3. **Performance:** Do either of the new strategies outperform random or uncertainty sampling?
4. **Label efficiency:** Which strategy achieves good performance with the fewest annotations?
5. **Redundancy:** Do any strategies repeatedly select very similar examples? Does this appear to affect learning?
6. **Stability:** How consistent are the results across repetitions?
7. **Project relevance:** Which acquisition strategy seems most appropriate for your application, and why?

Remember that **greater diversity or representativeness does not necessarily lead to better classification or prediction performance**.

A strategy may successfully optimize its acquisition objective without improving the model more than random sampling.

### 8.7 Document Your Experiment

Continue developing your project notebook as an **evolving experimental report**, rather than creating an isolated exercise.

In particular:

- **Build on your previous work.** Reuse your dataset, baseline model, active-learning loop, and evaluation procedures wherever possible.
- **Explain your design decisions.** Document your choice of representation, similarity measure, and acquisition functions.
- **Keep experiments reproducible.** Record random seeds, data splits, batch sizes, annotation budgets, and other relevant settings.
- **Present and interpret results.** Include learning curves, comparisons, and examples of selected data, accompanied by your observations.
- **Document limitations and open questions.** Identify where your acquisition functions might fail or where further experiments would be useful.

You now have several acquisition strategies to choose from. The goal is not to find a universally best algorithm, but to understand **which strategy is useful for your task, under which conditions, and why**.
